### Creating a spark session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("CBN_Silver_Transformation") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hdfs-namenode:9000") \
    .getOrCreate()

### Importing the datasets from the bronze (landing) layer

In [19]:
df_exch_raw = spark.read.option("multiLine", "true") \
    .json("hdfs://hdfs-namenode:9000/cbn_project/landing/raw_exchange_rates.json")

# Read the raw Inflation/CPI data
df_cpi_raw = spark.read.option("multiLine", "true") \
    .json("hdfs://hdfs-namenode:9000/cbn_project/landing/raw_inflation_rates.json")



### Cleaning the data to only include data from the last 20 years and only data for the NGN/USD currency pair

In [26]:
df_usd_silver = df_exch_raw.filter(F.col("currency") == "US DOLLAR") \
    .select(
        F.to_date(F.col("ratedate")).alias("rate_date"),
        F.col("buyingrate").cast("decimal(18,2)").alias("buying_rate"),
        F.col("sellingrate").cast("decimal(18,2)").alias("selling_rate"),
        F.col("centralrate").cast("decimal(18,2)").alias("central_rate")
    ) \
    .filter(
        F.col("rate_date") >= F.add_months(F.current_date(), -240)
    ) \
    .orderBy(F.col("rate_date").asc())

In [27]:
from pyspark.sql import functions as F

# Silver Logic: Clean and standardize CPI
df_cpi_silver = df_cpi_raw.select(
    # Create a proper date from 'period' or use tyear/tmonth
    F.to_date(F.concat(F.col("tyear"), F.lit("-"), F.col("tmonth"), F.lit("-01"))).alias("report_date"),
    F.col("allItemsYearOn").cast("decimal(10,2)").alias("headline_inflation"),
    F.col("foodYearOn").cast("decimal(10,2)").alias("food_inflation"),
    F.col("allItemsLessFrmProdAndEnergyYearOn").cast("decimal(10,2)").alias("core_inflation")
) \
.filter(F.col("report_date") >= F.add_months(F.current_date(), -240)) \
.orderBy(F.col("report_date").desc())



### Saving the cleaned and filtered data for use in the gold layer as parquet files

In [29]:

df_usd_silver.write.mode("overwrite").parquet("hdfs://hdfs-namenode:9000/cbn_project/silver/forex_daily")

# Save the monthly CPI
df_cpi_silver.write.mode("overwrite").parquet("hdfs://hdfs-namenode:9000/cbn_project/silver/cpi_monthly")